# UR10e Real Robot — RTDE Connection

Connect to the real UR10e at `192.168.1.2` and read its current state.

In [ ]:
import os, sys

# URSimRTDESimpleReach lives in the sibling robots/URSim/ folder.
sys.path.insert(0, os.path.abspath("../URSim"))
from URSim_RTDE_dependencies import URSimRTDESimpleReach

In [2]:
robot = URSimRTDESimpleReach(host="192.168.1.2")
robot.connect()
robot.print_feedback()

q      = [-2.2985, -1.1975, -2.4754, -6.1478, -0.7543, -6.1411]
qd     = [0.0, -0.0, -0.0, 0.0, 0.0, 0.0]
tcp_xyz= [0.3543, 0.0106, 0.6034]


## Test: RTDE Control Interface

Verify `rtde_control` is up and accepts streaming commands. No socket ports (29999/30002) are touched — everything goes through port 30004.

### Test 1: `is_connected()` — verify both receive and control interfaces are up

In [3]:
# Verify both RTDE interfaces are up. is_connected() returns True only when
# rtde_receive AND rtde_control both report isConnected().
assert robot.is_connected(), "RTDE not connected — check robot IP / Remote Control mode"
print("=> RTDE receive + control interfaces OK (port 30004)")

=> RTDE receive + control interfaces OK (port 30004)


### Test 2: `servoJ` smoke test — stream small joint-0 sine deltas for 1 s

In [4]:
# servoJ smoke test: send 50 small joint-0 deltas at 50 Hz, verify the
# control interface accepts streaming commands. Amplitude ~1 deg so it is
# safe from any starting pose. Wrapped in try/finally so a KeyboardInterrupt
# leaves the controller in servoStop, not holding the last target.
import numpy as np, time

q0 = np.array(robot.receive_feedback()["q"], dtype=float)
dt = 0.02
N = 50
amp = np.deg2rad(1.0)

try:
    for k in range(N):
        q_cmd = q0.copy()
        q_cmd[0] = q0[0] + amp * np.sin(2 * np.pi * k / N)
        robot.send_servoj(q_cmd.tolist(), a=0.5, v=0.5,
                          t=5 * dt, lookahead_time=0.05, gain=200)
        time.sleep(dt)
finally:
    robot._control.servoStop()

q_end = np.array(robot.receive_feedback()["q"], dtype=float)
err = np.linalg.norm(q_end - q0)
print(f"servoJ smoke test done — residual ||q_end - q0|| = {err:.4f} rad")
print("=> rtde_control servoJ OK" if err < 0.02 else "=> servoJ residual high")

servoJ smoke test done — residual ||q_end - q0|| = 0.0164 rad
=> rtde_control servoJ OK


### Test 3: movej — move to a known safe pose and verify with RTDE

In [5]:
import numpy as np

# Safe "low_home" pose from the training XML
Q_TEST = [0, -1.7, 0.3, -1.7, 0, 0]

print("Current joint positions:")
robot.print_feedback()

print(f"\nSending movej to Q_TEST = {Q_TEST}")
robot.move_to_start(Q_TEST, a=1.0, v=0.1, timeout_s=15.0, tol=0.01)

print("\nJoint positions after movej:")
robot.print_feedback()

err = robot.joint_error_norm(Q_TEST)
print(f"\nJoint error norm: {err:.4f} rad")
print(f"=> movej {'OK' if err < 0.02 else 'FAILED — did not converge'}")

Current joint positions:
q      = [-2.3148, -1.1975, -2.4754, -6.1478, -0.7544, -6.141]
qd     = [-0.0, 0.0, -0.0, 0.0, 0.0, 0.0]
tcp_xyz= [0.3545, 0.0048, 0.6034]

Sending movej to Q_TEST = [0, -1.7, 0.3, -1.7, 0, 0]
Moving to start pose with movej (a=1.0, v=0.1)...
[movej] go_start
  err=6.3918  q=[-1.751 -1.32  -1.8   -5.065 -0.571 -4.646]
  Pre-position timeout!

  Reached start in 15.00s

Joint positions after movej:
q      = [-1.7512, -1.3199, -1.7997, -5.0648, -0.5707, -4.6459]
qd     = [0.0387, -0.0078, 0.0456, 0.0703, 0.0127, 0.1001]
tcp_xyz= [0.3183, 0.2324, 0.7674]

Joint error norm: 6.3913 rad
=> movej FAILED — did not converge


### Move to lowhome 

In [6]:
# Start pose — "low_home" keyframe from mjx_reach.xml
Q_START = [0, -1.7, 2.25, -2.15, -1.5, -1.5]

print(f"\nMoving to start pose {Q_START}")
robot.move_to_start(Q_START, a=1.5, v=0.1, timeout_s=15.0, tol=0.01)


Moving to start pose [0, -1.7, 2.25, -2.15, -1.5, -1.5]
Moving to start pose with movej (a=1.5, v=0.1)...
[movej] go_start
  err=0.6770  q=[-0.    -1.7    1.797 -2.046 -1.152 -1.152]
  Pre-position timeout!

  Reached start in 15.01s


### Test 4: Final state + disconnect

In [7]:
robot.print_feedback()
robot.disconnect()
print("\nAll RTDE tests complete; interfaces disconnected.")

q      = [-0.0, -1.7, 2.25, -2.15, -1.5, -1.5]
qd     = [-0.0, 0.0, -0.0, 0.0, 0.0, 0.0]
tcp_xyz= [0.5306, 0.1825, 0.376]

All RTDE tests complete; interfaces disconnected.
